# One-hour-ahead Traffic Volume Forecasting on I-94 with Deep Learning
**Course:** Python for AI and Data Science (P4AIDS) — HCMUT · **Topic:** Demand Forecasting
**Dataset:** [Metro Interstate Traffic Volume — UCI ML Repository](https://archive.ics.uci.edu/dataset/492/metro+interstate+traffic+volume) (CC BY 4.0)

This notebook is the **deep learning part** of the group project. It uses the group's problem definition and data
contract, so its numbers can be compared directly with the machine learning models of the other members:
same rows, same chronological 70 / 15 / 15 split, same 46 numeric features + `weather_main`, same metrics.

| # | Requirement | Section |
|---|---|---|
| 1 | Problem Definition | §1 |
| 2 | Literature Review | §2 |
| 3 | Algorithm Implementation (preprocessing, training / fine-tuning with several seeds, model explanation) | §3 – §9 |
| 4 | Evaluation — compare and analyse the approaches | §10 – §12 |

```
Metro traffic/
├── data/                     raw file (+ data/processed/: the group's processed files, rebuilt if missing)
├── src/metro_traffic/        config · data · windows · baselines · metrics · models · training · tuning · explain · pipeline · artifacts · report
├── notebooks/                this notebook
├── train.py                  the same experiment from the command line
├── export_results.py         figures / tables / summary -> result/
├── dashboard/app.py          Streamlit dashboard (reads outputs/)
├── outputs/                  artifacts written in §12
└── result/                   exported figures, tables and summary
```

> **Google Colab:** upload the `Metro traffic` folder to Google Drive, set `PROJECT_DIR` in the first code cell,
> choose `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. LSTM + TCN with 10 Optuna trials each,
> 3 seeds and the ablation take roughly 1–2 hours on a T4. Set `run_tuning=False` / `run_ablation=False` for a
> faster run. Copy the group's `results/ml_predictions.parquet` into `ml_results/` to compare with the ML models.

## 0. Setup

In [ ]:
import os, sys, inspect

# Path of the "Metro traffic" folder on Google Drive -- change it if you put the folder elsewhere
PROJECT_DIR = "/content/drive/MyDrive/P4AIDS/Metro traffic"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install optuna
except ImportError:                      # running locally (e.g. Jupyter inside notebooks/)
    PROJECT_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
print("working directory:", os.getcwd())
print("project files:", sorted(os.listdir(".")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Code, display

from metro_traffic import data as D
from metro_traffic.config import Config
from metro_traffic.pipeline import Experiment, BASE_SUFFIX

pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

def show_source(obj):
    # Display the source code of a function / class from the package
    return Code(inspect.getsource(obj), language="python")

### Configuration
All settings are in one dataclass. `seeds` controls how many times every model is trained; the reported DL forecast
is the mean of the seed models.

In [ ]:
cfg = Config(
    raw_path="data/Metro_Interstate_Traffic_Volume.csv.gz",   # downloaded from UCI if missing
    processed_dir="data/processed",      # group processed files are used if present, otherwise rebuilt
    out_dir="outputs",
    models=("LSTM", "TCN"),              # the two DL architectures of docs/methodology.md
    seeds=(42, 43, 44),                  # every model is trained with all seeds (group: --seeds 3)
    max_epochs=80, patience=10,          # group protocol
    finetune_epochs=8,                   # fine-tuning on train + val at lr/5 (cosine)
    run_tuning=True, n_trials=10, tune_max_epochs=25, tune_patience=5,   # Optuna for both models
    run_ablation=True, ablation_seeds=(42, 43, 44),
    ml_results_dir="ml_results",         # group results/ml_predictions.parquet (optional) for the DM test
)
exp = Experiment(cfg)
cfg

## 1. Problem Definition

**Context.** The dataset records the hourly **westbound traffic volume on Interstate 94** between Minneapolis and
St Paul (MN DoT ATR station 301) from October 2012 to September 2018, with hourly weather (temperature, rain, snow,
cloud cover, weather type) and US holidays. Reliable short-term forecasts support traffic management, ramp metering
and travel-time information.

**Task (group definition).** At each hour *t*, predict the traffic volume of that hour, $y_t$, **one hour ahead**:

$$\hat y_t = f\big(\mathbf{x}_{t-167}, \dots, \mathbf{x}_t\big)$$

where row $\mathbf{x}_s$ only contains information available before $y_t$ is observed: traffic up to $t-1$ (lags,
rolling statistics, weekly profile), the calendar of $t$ and the weather observed at $t$ (nowcast setting).
The DL models read the last **168 hours (one week)** of these rows; the ML models of the group read only row $t$.

**Data contract shared with the ML part.**
- data from **2016-01-01** (the dense segment after the 2014–2015 sensor outage);
- chronological split by time span: **70 % train / 15 % validation / 15 % test**
  (train ≈ 2016-05 → 2018-01, validation → 2018-05, test 2018-05-22 → 2018-09-30);
- the target is never imputed; an hour is a sample only if its 168-hour history is fully observed;
- 46 numeric features + `weather_main`, statistics fitted on the training rows only.

**Metrics.** MAE (primary, vehicles/hour), RMSE, R², sMAPE and **MASE** (MAE divided by the in-sample MAE of the
seasonal-naive *t−24* forecast). Models are compared with the **Diebold–Mariano** test.

**Protocol.** All choices (architecture, hyper-parameters, epochs) use the validation set; the test set is used once
at the end.

## 2. Literature Review

**2.1 Classical statistical methods.** ARIMA/SARIMA (Box & Jenkins) and exponential smoothing are the traditional
tools for seasonal series; they assume linear dynamics and handle several nested seasonalities and many exogenous
variables poorly. Seasonal-naive forecasts (same hour yesterday / last week) are strong references for traffic.

**2.2 Machine learning on tabular features.** Ridge, Random Forest and gradient boosting on lag, rolling,
calendar and weather features — the approach of the ML members of the group.

**2.3 Deep learning for time series.**
- **Recurrent networks — LSTM** (Hochreiter & Schmidhuber, 1997) and **GRU** (Cho et al., 2014). Ma et al. (2015)
  applied LSTMs to traffic speed forecasting and outperformed ARIMA and SVR.
- **TCN** (Bai, Kolter & Koltun, 2018): dilated causal convolutions with a large receptive field.
- **Transformers** (Vaswani et al., 2017) and **PatchTST** (Nie et al., 2023), which turns patches of the series
  into tokens.
- **MLP-based models:** N-BEATS (Oreshkin et al., 2020); DLinear (Zeng et al., 2023) showed that simple models can
  rival Transformers → strong simple baselines are mandatory.
- Spatio-temporal graph networks (DCRNN, STGCN) target sensor networks; not applicable to a single station.

**2.4 Training practice.** Random initialisation changes the result of a network; reporting several seeds and
averaging their forecasts (a deep ensemble, Lakshminarayanan et al., 2017) gives a fairer and more stable estimate.
Hyper-parameters are searched with Optuna's TPE sampler (Akiba et al., 2019).

**2.5 Model explanation and comparison.** Integrated Gradients (Sundararajan et al., 2017), permutation importance
(Breiman, 2001) and occlusion explain which inputs and time lags a network uses. The Diebold–Mariano test
(Diebold & Mariano, 1995) checks whether two forecasts differ significantly.

**2.6 Our approach (group methodology).** (i) persistence, seasonal-naive and hour-of-week-mean baselines;
(ii) two tuned representatives of the sequence models — **LSTM** and **TCN** — on 168-hour windows;
(iii) Optuna tuning of both; (iv) every model trained with **seeds 42, 43, 44** and fine-tuned on train + validation;
(v) Integrated Gradients and weather occlusion; (vi) Diebold–Mariano test of the best ML vs the best DL model;
(vii) a cumulative feature-group ablation of the best DL model (post-merge plan).

## 3. Data: shared preprocessing
The group pipeline (`metro_traffic.data`, a line-by-line port of the group's `data.py`) cleans the raw file
(duplicated timestamps, holiday label spread over the whole day, impossible temperature / rain values), builds a
complete hourly grid, adds gap-aware lags and the engineered features, and splits the rows. If the group's
processed files are in `data/processed/` they are used directly; otherwise they are rebuilt with the same rules.

In [ ]:
exp.load_data()
frame, grid, meta = exp.frame, exp.grid, exp.meta
print("audit:", meta["audit"])
display(frame.groupby("split")["date_time"].agg(["min", "max", "size"]))

In [ ]:
show_source(D.clean)

**Key EDA facts used by the design.** Strong daily (7 h and 16–17 h weekday peaks) and weekly seasonality, very low
volume on holidays, weather effects much smaller than calendar effects, and a long sensor outage before 2016 (hence
the start date). The figure shows the modelled hours and the split.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
colors = {"train": "#1f4e9c", "val": "#e07b39", "test": "#2ca02c"}
for s, g in frame.groupby("split"):
    d = g.set_index("date_time")["traffic_volume"].resample("D").mean()
    ax.plot(d.index, d.values, color=colors[s], label=s)
ax.set_title("Daily mean traffic volume of the modelled hours"); ax.set_ylabel("vehicles/hour"); ax.legend()
plt.show()

## 4. Features
46 numeric columns, grouped as in the group ablation:

| Group | Columns | Idea |
|---|---|---|
| calendar | hour, dow, month, doy, weekend, holiday, Fourier terms | known for hour *t* |
| weather | temp_c, rain_1h, snow_1h, clouds_all, is_rain, is_snow (+ `weather_main` embedding) | observed at *t* |
| lags | lag_1/2/3/24/48/168, rolling mean/std over 3–168 h, diff_1 | traffic up to *t−1* |
| engineered | pre/post holiday, robust 4-week profile, hour-of-week profile | domain knowledge |

`how_mean` and `lag1_minus_how` use the training-set hour-of-week means (no future information, but fitted on
the training targets — set `drop_leaky_features=True` to remove them).

In [ ]:
groups = D.feature_groups(exp.numeric)
for g, cols in groups.items():
    print(f"{g:10s} ({len(cols):2d}): {cols}")

### 4.1 168-hour windows
A sample is a target hour $t$ of the model frame; its input is the block of grid rows $[t-167, t]$ of the 46
columns plus the `weather_main` code of each row. A few lag cells inside a window can be missing after a data gap;
they are filled with the **training median**, then every column is standardised with **training statistics**. The
target is standardised with the training mean and standard deviation. Windows are cut on the GPU by index, so the
$N \times 168 \times 46$ tensor is never stored.

In [ ]:
from metro_traffic.windows import build_windows
wd = exp.wd
print(f"grid rows {wd.X.shape[0]:,}, features {wd.X.shape[1]}, weather categories {wd.vocab}")
print(f"samples: train {len(wd.pos['train']):,} / val {len(wd.pos['val']):,} / test {len(wd.pos['test']):,}")
print(f"missing cells inside windows filled with the training median: {wd.nan_imputed:,}")
show_source(build_windows)

## 5. Baselines
- **Persistence:** $\hat y_t = y_{t-1}$
- **Seasonal naive:** $\hat y_t = y_{t-24}$ (also the MASE scale)
- **Hour-of-week mean:** training mean of the same hour of the week — the strongest naive reference.

In [ ]:
exp.run_baselines()
from metro_traffic.metrics import all_metrics
pd.DataFrame({n: all_metrics(exp.y["test"], p, exp.scale) for n, p in exp.preds["test"].items()}).T.round(3)

## 6. Deep learning models (group `dl.py`)
Both models read the numeric window plus an embedding of `weather_main` (size 4 or 8, tuned) and output the
standardised forecast of hour $t$:

| Model | Architecture |
|---|---|
| **LSTM** | embedding → 1–2 LSTM layers (32–128 hidden) → last hidden state → Linear(h, h/2) → ReLU → Dropout → Linear(h/2, 1) |
| **TCN** | embedding → 1×1 conv → 4 (1 layer) or 5 (2 layers) residual blocks of dilated causal convolutions (dilations 1, 2, 4, 8 (, 16), kernel 3; receptive field 61 / 125 h) → [last step, mean over time] → Linear(2h, h) → ReLU → Dropout → Linear(h, 1) |

In [ ]:
from metro_traffic import models, training
show_source(models.TCNRegressor)

### 6.1 Training protocol (docs/methodology.md §5)
- SmoothL1 loss (β = 0.5) on the standardised target, AdamW, ReduceLROnPlateau (factor 0.5, patience 2),
  gradient clipping at 1.0, early stopping on validation MAE (patience 10, at most 80 epochs), best epoch restored;
- **seeds 42, 43, 44** → three models per architecture, forecast = mean of the three;
- **fine-tuning:** each seed model continues for 8 epochs on train + validation at lr/5 with cosine decay;
  test metrics are reported before and after fine-tuning.

In [ ]:
show_source(training.train_model)

## 7. Hyper-parameter tuning (Optuna, both models)
TPE sampler (seed 42), **10 trials per architecture**, each trial trained with seed 42 for at most 25 epochs
(patience 5); objective = best validation MAE. Search space (shared): hidden {32, 64, 96, 128}, layers 1–2,
dropout 0–0.35, lr 1e-4–3e-3 (log), weight decay 1e-6–1e-3 (log), batch {128, 256, 512}, embedding {4, 8}.

In [ ]:
from metro_traffic.models import SEARCH_SPACE
print(SEARCH_SPACE)
exp.run_tuning()
exp.params

## 8. Final training with three seeds + fine-tuning
Each architecture is trained with its tuned settings and seeds 42, 43, 44, then every seed model is fine-tuned on
train + validation. **Selection** of the best DL model uses the validation MAE of the ensemble *before*
fine-tuning (after fine-tuning the validation set is no longer held out).

In [ ]:
exp.run_final()
exp.select_best()
seeds = pd.DataFrame(exp.seed_rows)
display(seeds.round(1))
summary = seeds.groupby("model").agg(val_mean=("val_MAE", "mean"), val_std=("val_MAE", "std"),
                                     test_base_mean=("test_MAE_no_finetune", "mean"),
                                     test_ft_mean=("test_MAE", "mean"), test_ft_std=("test_MAE", "std"))
summary["val_ensemble"] = [exp.final[m]["val_MAE_ensemble"] for m in summary.index]
summary.sort_values("val_ensemble").round(1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for k, h in exp.histories.items():
    if "fine-tune" in k:
        continue
    h = pd.DataFrame(h)
    ax[0].plot(h.epoch, h.val_mae, label=k, alpha=0.8)
    ax[1].plot(h.epoch, h.lr, alpha=0.8)
ax[0].set(title="Validation MAE per epoch", xlabel="epoch")
ax[1].set(title="Learning rate", xlabel="epoch", yscale="log")
ax[0].legend(fontsize=8)
plt.show()
exp.best

### 8.1 Ablation: which feature groups matter? (post-merge plan 4.3)
The selected model is retrained (all seeds, train only) on cumulative groups: calendar → + weather → + lags →
+ engineered (= full model). Without the weather group the `weather_main` embedding is removed as well. The group
report runs the same ablation with LightGBM, so ML and DL can be compared stage by stage.

In [ ]:
exp.run_ablation()
if len(exp.ablation):
    display(exp.ablation.groupby(["stage", "features"])[["val_MAE", "test_MAE"]].agg(["mean", "std"]).round(1))

## 9. Model explanation (docs/methodology.md §7)
- **Integrated Gradients:** 256 random test windows, 24 integration steps, zero baseline in standardised space;
  attributions aggregated by feature and by lag position. Computed for the fine-tuned seed ensemble (IG is linear
  in the output, so it is the mean of the per-seed IGs).
- **Weather occlusion:** replace each weather category with the unknown token and measure the mean absolute
  change in prediction.

In [ ]:
exp.run_explain()
fig, ax = plt.subplots(1, 2, figsize=(16, 5.5))
top = exp.ig_features.head(15).iloc[::-1]
ax[0].barh(top.feature, top.mean_abs_attr, color="#1f4e9c"); ax[0].set_title("Integrated Gradients: top 15 features")
ax[1].plot(exp.ig_lag.lag, exp.ig_lag.mean_abs_ig, color="#1f4e9c")
ax[1].set(title="Attribution over the 168-hour window", xlabel="lag relative to the target hour")
plt.show()
display(exp.occlusion.round(2))

## 10. Evaluation on the test set (2018-05-22 → 2018-09-30)

In [ ]:
exp.load_ml_predictions()
exp.run_dm()
res = exp.results_table()
test = res[res.split == "test"].drop(columns="split").set_index("model").sort_values("MAE")
display(test.style.format({"MAE": "{:.1f}", "RMSE": "{:.1f}", "R2": "{:.4f}", "sMAPE": "{:.2f}", "MASE": "{:.3f}"}))

### 10.1 Diebold–Mariano tests (A = best DL; the primary test is against the best ML model on validation)

In [ ]:
exp.dm.round(4)

### 10.2 Forecast examples and error analysis

In [ ]:
BEST = exp.best["model"]
t = pd.DatetimeIndex(exp.wd.times["test"]); y = exp.y["test"]
from metro_traffic.report import best_window_start
s = best_window_start(t, 14); full = pd.date_range(s, s + pd.Timedelta(days=14), freq="h", inclusive="left")
pos = pd.Series(np.arange(len(t)), index=t).reindex(full); ok = pos.notna().values; ii = pos.fillna(0).astype(int).values
seg = lambda v: np.where(ok, np.asarray(v)[ii], np.nan)
plt.figure(figsize=(16, 4.5))
plt.plot(full, seg(y), "k", lw=2, label="actual")
plt.plot(full, seg(exp.preds["test"][BEST]), lw=1.6, label=BEST)
plt.plot(full, seg(exp.preds["test"]["Hour-of-week mean"]), "--", lw=1.2, label="Hour-of-week mean")
if exp.best_ml:
    plt.plot(full, seg(exp.preds["test"][exp.best_ml]), lw=1.2, label=exp.best_ml)
plt.legend(); plt.title("One-hour-ahead forecasts on two test weeks"); plt.show()

part = exp.frame[exp.frame.split == "test"]
err = pd.DataFrame({"hour": t.hour, "day_type": np.select([part.is_holiday.values == 1, part.is_weekend.values == 1],
                                                          ["Holiday", "Weekend"], "Weekday"),
                    BEST: np.abs(exp.preds["test"][BEST] - y),
                    "Hour-of-week mean": np.abs(exp.preds["test"]["Hour-of-week mean"] - y)})
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
err.groupby("hour")[[BEST, "Hour-of-week mean"]].mean().plot(ax=ax[0], marker="o", title="MAE by hour")
err.groupby("day_type")[[BEST, "Hour-of-week mean"]].mean().plot.bar(ax=ax[1], title="MAE by day type", rot=0)
plt.show()

## 11. Automatic summary

In [ ]:
b = test.loc[BEST]; ref = test.loc["Hour-of-week mean"]
sd = pd.DataFrame(exp.seed_rows).query("model == @BEST")
print(f"Selected DL model: {BEST} ({'Optuna-tuned' if exp.best['tuned'] else 'default settings'}), seeds {list(cfg.seeds)}")
nb = exp.best["test_no_finetune"]
print(f"Before fine-tuning: test MAE {nb['MAE']:.1f}")
print(f"Test MAE {b.MAE:.1f} | RMSE {b.RMSE:.1f} | R2 {b.R2:.3f} | sMAPE {b.sMAPE:.2f} % | MASE {b.MASE:.3f}")
print(f"Single seeds: test MAE {sd.test_MAE.mean():.1f} ± {sd.test_MAE.std(ddof=0):.1f}; ensemble {b.MAE:.1f}")
print(f"vs Hour-of-week mean (MAE {ref.MAE:.1f}): {100 * (1 - b.MAE / ref.MAE):.1f} % lower MAE")
if exp.best_ml:
    m = test.loc[exp.best_ml]; d = exp.dm[exp.dm.primary].iloc[0]
    print(f"Best ML {exp.best_ml}: test MAE {m.MAE:.1f}; DM statistic {d.dm_stat:.2f}, p = {d.p_value:.2g}")
print("Group reference run: TCN 126.6, LSTM 131.2, lightgbm_blend 119.7 (test MAE)")

## 12. Save artifacts for the dashboard
`exp.save()` writes metrics, predictions (validation + test, every seed of the selected model), histories, the
tuning log, the ablation, explanations, the DM tests and the hourly data to `outputs/`, plus the weights of the
fine-tuned seed models. The Streamlit dashboard (`streamlit run dashboard/app.py`) only reads this folder.

In [ ]:
exp.save()
sorted(os.listdir(cfg.out_dir))

### 12.1 Export figures, tables and a summary to `result/`

In [ ]:
from metro_traffic.artifacts import load_artifacts
from metro_traffic.report import export_results
files = export_results(load_artifacts(cfg.out_dir), "result")
print(len(files), "files written to result/")

## 13. Discussion and conclusion

*(Read the numbers printed in §10–§11 and adjust the statements below to match your run.)*

**One-hour-ahead is a strong-baseline problem.** The hour-of-week mean and the weekly naive forecast already capture
most of the daily and weekly pattern; persistence and the seasonal-naive *t−24* forecast are weaker because they
copy the wrong part of the cycle (last hour, or yesterday even when yesterday was a weekend). A useful model must
beat the hour-of-week mean, not only the naive *t−24* forecast used as the MASE scale.

**LSTM vs TCN.** Both combine the recent traffic level (lags, rolling statistics) with the calendar and weekly
profile. The TCN sees the whole window through dilated convolutions and mean pooling; the LSTM compresses it into
the last hidden state. In the group run the TCN was better on validation (161.6 vs 173.5) and test (126.6 vs
131.2). Differences should be judged against the seed-to-seed spread and with the Diebold–Mariano test.

**DL vs ML.** On a single detector with ~9k training hours and strong hand-made features, tuned gradient boosting
is a very strong competitor; the group run found the LightGBM blend significantly better than the TCN
(DM p ≈ 3e-5). Compare your numbers with §11.

**Seeds and fine-tuning.** The ensemble of three seeds is at least as accurate as the average single seed and removes
the dependence on one lucky initialisation. Fine-tuning on train + validation lets the models see the months right
before the test period; its effect is reported per encoder in the dashboard.

**Explanation.** IG and the ablation show which inputs matter: traffic history (lags) and the calendar /
engineered weekly profile dominate; weather adds little on average, and the occlusion shows which weather
categories move the prediction most.

**Limitations.** Weather at *t* is the observed value (nowcast); in production it would be a weather forecast.
Single station, outage before 2016, unusual events not in the data, and a single chronological test period.

**Future work.** More architectures (GRU, PatchTransformer — post-merge plan 4.2), rolling-origin evaluation,
quantile loss for prediction intervals, more seeds, an ML + DL ensemble.

## References
1. Hogue, J. (2019). *Metro Interstate Traffic Volume* [Dataset]. UCI Machine Learning Repository.
2. Box, G. E. P., & Jenkins, G. M. (1970). *Time Series Analysis: Forecasting and Control*.
3. Hochreiter, S., & Schmidhuber, J. (1997). Long Short-Term Memory. *Neural Computation*, 9(8).
4. Cho, K. et al. (2014). Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation. *EMNLP*.
5. Ma, X. et al. (2015). Long short-term memory neural network for traffic speed prediction using remote microwave sensor data. *Transportation Research Part C*, 54.
6. Bai, S., Kolter, J. Z., & Koltun, V. (2018). An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling. *arXiv:1803.01271*.
7. Vaswani, A. et al. (2017). Attention Is All You Need. *NeurIPS*.
8. Oreshkin, B. et al. (2020). N-BEATS. *ICLR*.
9. Nie, Y. et al. (2023). A Time Series is Worth 64 Words: Long-term Forecasting with Transformers (PatchTST). *ICLR*.
10. Zeng, A. et al. (2023). Are Transformers Effective for Time Series Forecasting? *AAAI*.
11. Lakshminarayanan, B., Pritzel, A., & Blundell, C. (2017). Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles. *NeurIPS*.
12. Akiba, T. et al. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. *KDD*.
13. Sundararajan, M., Taly, A., & Yan, Q. (2017). Axiomatic Attribution for Deep Networks. *ICML*.
14. Breiman, L. (2001). Random Forests. *Machine Learning*, 45.
15. Diebold, F. X., & Mariano, R. S. (1995). Comparing Predictive Accuracy. *Journal of Business & Economic Statistics*, 13(3).